# CCS4354 — Tensors and Graphs
# Group Coursework: Node Classification on the OGBN-Arxiv Citation Network

**Module:** CCS4354 — Tensors and Graphs

**Dataset:** OGBN-Arxiv (Open Graph Benchmark) — a citation network of ~169,000 arXiv Computer
Science papers, ~1.1 million citation edges, 128-dimensional node features, 40 subject-area classes.

**Task:** Node Classification — predict the subject category of every paper using its text
features **and** the citation graph structure around it.

---

### How this notebook is organised

Every task from the coursework brief has its own numbered section. Inside each section, every
single code cell is preceded by a short markdown cell that explains **what** the cell does and
**why** it is needed — as requested, this is a "one explanation per cell" notebook, not a wall of
code with a single paragraph at the top.

| Section | Coursework Task |
|---|---|
| 1 | Task 01 — Tensor Fundamentals |
| 2 | Task 02 — Graph Representation & Analysis |
| 3 | Task 03 — Graph Data Preparation |
| 4 | Task 04 — GNN Development (GCN + GAT) |
| 5 | Task 05 — Model Training & Optimization |
| 6 | Task 06 — Model Evaluation |
| 7 | Task 07 — Explainability & Embedding Analysis |
| 8 | Task 08 — Export artifacts for the Streamlit Dashboard |
| 9 | Task 09 — Notes for the Technical Report & Viva |

> **Note on running this notebook:** the first time you run Section 2 the `ogb` library will
> download the OGBN-Arxiv dataset (~90&nbsp;MB compressed) from the official OGB server. This
> requires an active internet connection. Everything downstream (splits, models, dashboard
> artifacts) is deterministic given the same `random_seed`, so results are reproducible.


## Five-member development workflow

This is the clean shared notebook. All original implementation cells and outputs have been
removed. Each member creates code cells only beneath their assigned task headings and pushes the
same notebook from their own feature branch.

1. Member 1 — Tasks 01–02: tensors and graph analysis.
2. Member 2 — Task 03 plus the GCN architecture.
3. Member 3 — GAT plus Task 05 training and optimisation.
4. Member 4 — Tasks 06–07 evaluation and explainability.
5. Member 5 / Project Owner — dashboard prototype, extended work and final integration.

The fully executed original remains under `notebook/reference/` and must not be edited or pushed
as the final notebook. Because `.ipynb` files can conflict easily, members should pull the latest
`main` immediately before starting and avoid changing another member's markdown or cells.


## 0. Environment Setup

Before touching tensors or graphs we install and import every library the brief asks for:
PyTorch (deep learning framework), PyTorch Geometric + OGB (graph learning), and the supporting
stack — NumPy, Pandas, Matplotlib, Scikit-Learn, NetworkX. We also fix random seeds so that every
group member gets the same numbers when they re-run the notebook.

In [ ]:
# Install the graph-learning stack. Safe to re-run — pip skips packages that are already installed.
# Google Colab already provides a CUDA-enabled PyTorch build. We preserve it instead of
# replacing it with a CPU-only wheel, then install the remaining libraries around it.
import os, sys, subprocess
# Must be set before importing torch so Colab's CUDA allocator picks it up.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

pip_install("ogb", "pandas", "matplotlib", "scikit-learn", "networkx", "seaborn")

try:
    import torch
except ImportError:
    pip_install("torch")
    import torch
print("Torch version:", torch.__version__)

# torch_geometric ships pure-python wheels on PyPI for recent torch versions, so a plain
# install works in almost all environments (Colab, local CPU/GPU, university lab machines).
pip_install("torch_geometric")


**Package installation above may print a lot of pip output above — that's expected.**
Next, we import every module used throughout the notebook in one place, and fix random seeds for
reproducibility (so re-running the notebook gives the same numbers every time) and pick the
compute device (GPU if available, otherwise CPU).

In [ ]:
# Core imports used throughout the whole notebook.
import os
import json
import time
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Reproducibility: every group member should get identical numbers with this seed.
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# All dashboard-ready artifacts (metrics, embeddings, predictions) get written here so that
# app.py (the Streamlit dashboard built for Task 08) can read them without re-running training.
ARTIFACT_DIR = os.path.join("dashboard", "artifacts")
os.makedirs(ARTIFACT_DIR, exist_ok=True)

def free_gpu_memory(verbose=True):
    """Release Python/IPython references and cached CUDA blocks between heavy runs."""
    import gc
    try:
        ip = get_ipython()
        ip.displayhook.cache_size = 0
        Out.clear()
        for name in ("_", "__", "___"):
            if name in globals():
                globals()[name] = None
    except Exception:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        if verbose:
            print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated, "
                  f"{torch.cuda.memory_reserved()/1e9:.2f} GB reserved")


---
## 1. Task 01 — Tensor Fundamentals

> **Owner: Member 1** — tensor fundamentals. Add your implementation code cells in this section.

A **tensor** is simply a multi-dimensional array with a fixed data type — a scalar is a 0-D
tensor, a vector is 1-D, a matrix is 2-D, and a batch of feature matrices (like the node features
of a graph) is 3-D and beyond. Every operation a Graph Neural Network performs later in this
notebook — message passing, attention, pooling — is built out of the primitive tensor operations
demonstrated below. This section proves each primitive works before we rely on it inside a model.

### 1.1 Tensor Creation

We create tensors in the three most common ways: from raw Python data, from NumPy arrays, and
using PyTorch's built-in constructors (`zeros`, `ones`, `rand`, `arange`). We also inspect a
tensor's `shape`, `dtype`, and `device`, since these three properties are the first thing to check
whenever a GNN throws a shape-mismatch error.

In [ ]:
# 1.1 Tensor creation ----------------------------------------------------------------------
t_from_list  = torch.tensor([[1, 2, 3], [4, 5, 6]])                 # from nested python lists
t_from_numpy = torch.from_numpy(np.array([1.0, 2.0, 3.0]))          # from a numpy array
t_zeros      = torch.zeros((3, 4))                                  # 3x4 matrix of zeros
t_ones       = torch.ones((2, 2))                                   # 2x2 matrix of ones
t_rand       = torch.rand((2, 3))                                   # uniform random in [0, 1)
t_arange     = torch.arange(0, 10, 2)                                # like python range()

print("From list:\n", t_from_list)
print("Shape:", t_from_list.shape, "| dtype:", t_from_list.dtype, "| device:", t_from_list.device)
print("\nFrom numpy:", t_from_numpy)
print("Zeros:\n", t_zeros)
print("Random:\n", t_rand)
print("Arange:", t_arange)


### 1.2 Tensor Indexing

Node feature matrices in a graph have shape `[num_nodes, num_features]`. Selecting a single
paper's feature vector, a slice of papers, or filtering papers by a boolean mask are exactly the
indexing patterns used later when we split the arXiv graph into train/validation/test sets.

In [ ]:
# 1.2 Tensor indexing -----------------------------------------------------------------------
features = torch.arange(20).reshape(5, 4).float()   # pretend: 5 "papers", 4 "features" each
print("Full feature matrix:\n", features)

print("\nRow 0 (single paper's features):", features[0])
print("Column 1 (one feature across all papers):", features[:, 1])
print("Rows 1-3 (a slice of papers):\n", features[1:4])

mask = features[:, 0] > 4                            # boolean mask, like selecting "train" nodes
print("\nBoolean mask (feature 0 > 4):", mask)
print("Filtered rows:\n", features[mask])

idx = torch.tensor([0, 2, 4])                          # fancy/advanced indexing
print("\nFancy-indexed rows (papers 0, 2, 4):\n", features[idx])


### 1.3 Tensor Reshaping

GNN layers frequently need to reshape tensors — for example flattening a batch of attention heads
back into a single feature vector, or turning a flat edge array into a `[2, num_edges]` matrix
(the exact format PyTorch Geometric expects for `edge_index`). `view`, `reshape`, `flatten`,
`squeeze` and `unsqueeze` are demonstrated here.

In [ ]:
# 1.3 Tensor reshaping ----------------------------------------------------------------------
x = torch.arange(12)
print("Original 1-D tensor:", x, "shape:", x.shape)

x_reshaped = x.reshape(3, 4)
print("\nReshaped to (3, 4):\n", x_reshaped)

x_view = x.view(4, 3)                      # view shares memory with x, reshape may copy
print("\nView as (4, 3):\n", x_view)

x_flat = x_reshaped.flatten()
print("\nFlattened back to 1-D:", x_flat)

# unsqueeze/squeeze: exactly what we use to turn a flat edge list [E] into edge_index [2, E]
edge_flat = torch.tensor([0, 1, 1, 2, 2, 0])          # 3 edges, flattened as (src, dst, src, dst..)
edge_index = edge_flat.reshape(-1, 2).t().contiguous() # -> shape [2, num_edges], PyG's format
print("\nFlat edge list:", edge_flat)
print("Reshaped into PyG-style edge_index [2, num_edges]:\n", edge_index)

col = torch.tensor([1, 2, 3])
print("\nBefore unsqueeze:", col.shape, "-> after unsqueeze(1):", col.unsqueeze(1).shape)
print("Squeeze removes size-1 dims:", col.unsqueeze(1).squeeze().shape)


### 1.4 Matrix Multiplication

Every GNN layer is, at its core, a matrix multiplication: `H' = A_hat @ H @ W`, where `A_hat` is
a (normalized) adjacency matrix, `H` is the node feature matrix, and `W` is a learnable weight
matrix. We demonstrate `@` / `torch.matmul` for 2-D matrices, batched matrix multiplication for
3-D tensors, and the difference between matrix multiplication and elementwise multiplication.

In [ ]:
# 1.4 Matrix multiplication -----------------------------------------------------------------
A = torch.rand(4, 3)     # e.g. 4 papers, 3 features
W = torch.rand(3, 2)     # a "weight matrix" projecting 3 features down to 2

H = A @ W                # equivalent to torch.matmul(A, W)
print("A (4x3) @ W (3x2) -> H (4x2):\n", H)

# Elementwise multiplication is NOT the same as matrix multiplication - shapes must match exactly.
elementwise = torch.tensor([1., 2., 3.]) * torch.tensor([4., 5., 6.])
print("\nElementwise multiply (Hadamard product):", elementwise)

# Batched matmul: e.g. attention scores computed independently per attention "head"
batch_A = torch.rand(8, 4, 3)   # 8 heads, 4 papers, 3 features
batch_W = torch.rand(8, 3, 2)   # 8 heads, each with its own 3x2 projection
batch_H = torch.bmm(batch_A, batch_W)
print("\nBatched matmul result shape (8 heads, 4 nodes, 2 out-features):", batch_H.shape)


### 1.5 Tensor Broadcasting

Broadcasting lets PyTorch apply an operation between tensors of different (but compatible) shapes
without manually copying data — for example adding a single `bias` vector to every row of a
feature matrix, or normalizing every node's feature vector by that node's own degree. GNN message
passing relies on broadcasting constantly, so we build intuition for the broadcasting rules here.

In [ ]:
# 1.5 Tensor broadcasting -------------------------------------------------------------------
feature_matrix = torch.rand(5, 4)     # 5 papers, 4 features
bias = torch.rand(4)                  # a single bias vector, shape (4,)

# Broadcasting rule: bias (4,) is stretched to (5, 4) to match feature_matrix, then added.
biased = feature_matrix + bias
print("feature_matrix (5,4) + bias (4,) -> shape:", biased.shape)

# A very GNN-specific example: dividing every node's row by that node's own degree (normalization)
degree = torch.tensor([1., 2., 3., 4., 5.]).unsqueeze(1)     # shape (5, 1)
normalized = feature_matrix / degree                          # (5,4) / (5,1) broadcasts over columns
print("\nDegree-normalized features:\n", normalized)

try:
    torch.rand(5, 4) + torch.rand(3, 4)     # incompatible shapes -> intentionally raises
except RuntimeError as e:
    print("\nExpected broadcasting error when shapes are incompatible:\n", e)


### 1.6 Tensor Aggregation Operations

Message passing in a GNN ends with an **aggregation** step: every node collects its neighbours'
messages and combines them with sum, mean, or max. We rehearse those exact aggregation primitives
here (`sum`, `mean`, `max`, `min`, `std`) along the node dimension, plus `argmax`, which is what
turns a model's output logits into a predicted class label.

In [ ]:
# 1.6 Tensor aggregation operations ---------------------------------------------------------
scores = torch.rand(6, 40)     # pretend: 6 papers, 40-class logits (arXiv has 40 subject areas)

print("Sum over classes (per paper):", scores.sum(dim=1)[:3])
print("Mean over papers (per class):", scores.mean(dim=0)[:5])
print("Max value per paper:", scores.max(dim=1).values)
print("Predicted class per paper (argmax):", scores.argmax(dim=1))
print("Std-dev over classes (per paper):", scores.std(dim=1)[:3])

# This is literally the "aggregate neighbour messages" step used inside GraphSAGE / GCN:
neighbour_messages = torch.rand(4, 3, 8)   # 4 target nodes, 3 neighbours each, 8-dim messages
aggregated_mean = neighbour_messages.mean(dim=1)   # mean-aggregate over the neighbour dimension
aggregated_max, _ = neighbour_messages.max(dim=1)  # max-aggregate over the neighbour dimension
print("\nMean-aggregated neighbour messages shape:", aggregated_mean.shape)
print("Max-aggregated neighbour messages shape:", aggregated_max.shape)


### 1.7 GPU Tensor Operations

If a CUDA-enabled GPU is available, tensors and models should be moved onto it with `.to(device)`
for a large speed-up on a graph this size (169k nodes / 1.1M edges). This cell checks for GPU
availability and demonstrates moving a tensor to the GPU and back; if no GPU is present it falls
back gracefully to CPU so the notebook still runs end-to-end on any machine.

In [ ]:
# 1.7 GPU tensor operations -----------------------------------------------------------------
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU device name:", torch.cuda.get_device_name(0))

cpu_tensor = torch.rand(1000, 1000)

start = time.time()
_ = cpu_tensor @ cpu_tensor
cpu_time = time.time() - start
print(f"1000x1000 matmul on CPU: {cpu_time*1000:.2f} ms")

if torch.cuda.is_available():
    gpu_tensor = cpu_tensor.to(DEVICE)
    torch.cuda.synchronize()
    start = time.time()
    _ = gpu_tensor @ gpu_tensor
    torch.cuda.synchronize()
    gpu_time = time.time() - start
    print(f"1000x1000 matmul on GPU: {gpu_time*1000:.2f} ms  (device: {gpu_tensor.device})")
    back_to_cpu = gpu_tensor.to("cpu")
    print("Moved back to CPU:", back_to_cpu.device)
else:
    print("No GPU detected in this environment - all later training runs on CPU (DEVICE =",
          DEVICE, "). On a CUDA machine, every .to(DEVICE) call below automatically uses the GPU.")


---
## 2. Task 02 — Graph Representation and Analysis

> **Owner: Member 1** — graph representation and structural analysis.

This section loads the real OGBN-Arxiv graph via the official `ogb` loader, represents it as an
edge list, visualizes a manageable sample sub-graph (the full graph has 1.1M edges — plotting all
of it would be an unreadable black smudge), describes the node features, and runs the three
structural analyses the brief asks for: degree distribution, graph density, and connected
components.

### 2.1 Load the OGBN-Arxiv Dataset

`PygNodePropPredDataset` downloads (once, then caches locally) and loads the dataset directly into
a PyTorch Geometric `Data` object — this single object holds the node features (`x`), the edges
(`edge_index`), the paper labels (`y`), and the official train/valid/test split provided by OGB
so that every group's results are comparable on the same split.

In [ ]:
# 2.1 Load the OGBN-Arxiv dataset -------------------------------------------------------------
from ogb.nodeproppred import PygNodePropPredDataset

# --- Compatibility fix for PyTorch >= 2.6 -----------------------------------------------------
# PyTorch 2.6 changed torch.load()'s default from weights_only=False to weights_only=True.
# The ogb package's cached/processed dataset file was written with torch.save() before this
# change, so loading it now raises "Weights only load failed / Unsupported global:
# torch_geometric.data.data.DataEdgeAttr". The fix is to explicitly allow-list the small set of
# PyG classes the cached file contains -- this is safe because we generated/downloaded this file
# ourselves from the official OGB server in the previous cell.
import torch
try:
    from torch_geometric.data.data import DataEdgeAttr, DataTensorAttr
    from torch_geometric.data.storage import GlobalStorage
    torch.serialization.add_safe_globals([DataEdgeAttr, DataTensorAttr, GlobalStorage])
except ImportError:
    pass  # older torch_geometric versions don't need this at all

dataset = PygNodePropPredDataset(name="ogbn-arxiv", root="data/")
data = dataset[0]                      # the single graph object (this dataset has exactly one graph)
split_idx = dataset.get_idx_split()    # official OGB train / valid / test node indices

print(data)
print("\nNumber of nodes:", data.num_nodes)
print("Number of edges:", data.num_edges)
print("Node feature dimension:", data.x.shape[1])
print("Number of classes:", dataset.num_classes)
print("Train / Valid / Test sizes:",
      len(split_idx["train"]), len(split_idx["valid"]), len(split_idx["test"]))


### 2.2 Represent the Graph as an Edge List

PyG stores edges as a `[2, num_edges]` tensor (`edge_index`), which is already an edge list in
matrix form — row 0 is source papers, row 1 is destination (cited) papers. We convert it into a
human-readable Pandas DataFrame `(source, target)` edge list, which is the representation asked
for in the brief and is also the easiest format to hand to NetworkX for visualization.

In [ ]:
# 2.2 Represent the graph as an edge list ------------------------------------------------------
edge_index_np = data.edge_index.numpy()
edge_list_df = pd.DataFrame({
    "source_paper": edge_index_np[0],
    "target_paper_cited": edge_index_np[1],
})
print("Total directed citation edges:", len(edge_list_df))
edge_list_df.head(10)


### 2.3 Visualize a Sample Subgraph

Plotting all 169,000 nodes is unreadable and slow, so we take an ego-network around a "seed"
paper and draw that instead — a common and honest way to visualize a large graph's *local*
structure. Two details matter for this to actually work:

1. **The seed needs a moderate degree.** A seed with 0 citations produces an empty picture; a
   seed with thousands of citations (a landmark paper) produces an unreadable hairball. We sample
   the seed from papers with a moderate number of citations instead of a fully random paper.
2. **Capping the subgraph's size must preserve connectivity.** Simply keeping a *random* subset of
   the nodes almost always breaks every edge (both endpoints must survive the random cut, which is
   unlikely). We instead grow the subgraph with a breadth-first search that stops once it reaches
   the node budget — every node it adds is, by construction, already connected to the graph.

In [ ]:
# 2.3 Visualize a sample subgraph ---------------------------------------------------------------
row_full, col_full = data.edge_index
undirected_row = torch.cat([row_full, col_full])   # treat citations as undirected for exploration
undirected_col = torch.cat([col_full, row_full])

# Pick a seed paper with a moderate number of citers (5-40) - avoids both "empty picture"
# (0-citation seed) and "unreadable hairball" (a landmark paper with thousands of citations).
in_degree_all = torch.bincount(col_full, minlength=data.num_nodes)
moderate_degree_nodes = ((in_degree_all >= 5) & (in_degree_all <= 40)).nonzero(as_tuple=True)[0]
torch.manual_seed(RANDOM_SEED)
seed_node = int(moderate_degree_nodes[torch.randint(0, moderate_degree_nodes.size(0), (1,))].item())

def bfs_capped_subgraph(seed, max_nodes=90, num_hops=2):
    """Breadth-first expansion that always returns a CONNECTED node set, capped at max_nodes.
    Unlike randomly sub-sampling nodes from a large neighbourhood, BFS only ever adds nodes that
    are directly reachable from what's already in the set, so edges between kept nodes always
    survive."""
    visited = {seed}
    frontier = {seed}
    for _ in range(num_hops):
        if len(visited) >= max_nodes or not frontier:
            break
        frontier_t = torch.tensor(list(frontier))
        mask = torch.isin(undirected_row, frontier_t)
        neighbours = undirected_col[mask].unique().tolist()
        new_nodes = [n for n in neighbours if n not in visited]
        budget = max_nodes - len(visited)
        new_nodes = new_nodes[:budget]
        visited.update(new_nodes)
        frontier = set(new_nodes)
    return visited

visited_nodes = bfs_capped_subgraph(seed_node, max_nodes=90, num_hops=2)
visited_tensor = torch.tensor(sorted(visited_nodes))

# Keep the ORIGINAL directed edges (so drawn arrows still show real citation direction) among
# the visited node set only.
edge_mask = torch.isin(row_full, visited_tensor) & torch.isin(col_full, visited_tensor)
sub_edge_index = data.edge_index[:, edge_mask]

G_sample = nx.DiGraph()
G_sample.add_nodes_from(visited_tensor.tolist())      # keep low-degree nodes visible even if isolated within the sample
G_sample.add_edges_from(sub_edge_index.t().tolist())

plt.figure(figsize=(9, 7))
pos = nx.spring_layout(G_sample, seed=RANDOM_SEED, k=0.45)
nx.draw_networkx_nodes(G_sample, pos, node_size=70, node_color="#4C72B0", alpha=0.85)
nx.draw_networkx_edges(G_sample, pos, arrows=True, arrowsize=8, alpha=0.45, width=0.9,
                        connectionstyle="arc3,rad=0.05")
plt.title(f"2-hop ego-network around paper #{seed_node}  ({G_sample.number_of_nodes()} nodes,"
          f" {G_sample.number_of_edges()} citation edges)")
plt.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(ARTIFACT_DIR, "sample_subgraph.png"), dpi=150)
plt.show()


### 2.4 Describe the Node Features

Each paper's feature vector is a 128-dimensional embedding computed by averaging Word2Vec
embeddings of the words in its title and abstract (this is how OGB pre-processed the raw text —
we don't need to redo it). We inspect the value range, mean, and standard deviation of the raw
features to understand what kind of preprocessing (Task 03) they will need.

In [ ]:
# 2.4 Describe the node features ------------------------------------------------------------
x = data.x
print("Feature matrix shape:", x.shape, " (num_papers x embedding_dim)")
print("Value range: [{:.4f}, {:.4f}]".format(x.min().item(), x.max().item()))
print("Per-feature mean (first 10 dims):", x.mean(dim=0)[:10].numpy().round(4))
print("Per-feature std  (first 10 dims):", x.std(dim=0)[:10].numpy().round(4))
print("Any missing values (NaN)?", bool(torch.isnan(x).any()))

feature_summary = pd.DataFrame({
    "mean": x.mean(dim=0).numpy(),
    "std": x.std(dim=0).numpy(),
    "min": x.min(dim=0).values.numpy(),
    "max": x.max(dim=0).values.numpy(),
})
feature_summary.describe()


### 2.5 Degree Distribution Analysis

The **degree** of a node is how many citation edges touch it. Citation networks are famously
*scale-free* — most papers cite/are-cited a handful of times, while a small number of landmark
papers accumulate huge numbers of citations. We compute in-degree (times cited) and out-degree
(papers cited) for every node and plot the distribution on a log-log scale, which is the standard
way to reveal a power-law/scale-free pattern.

In [ ]:
# 2.5 Degree distribution analysis ------------------------------------------------------------
row, col = data.edge_index
out_degree = torch.bincount(row, minlength=data.num_nodes)   # papers this paper cites
in_degree  = torch.bincount(col, minlength=data.num_nodes)   # times this paper is cited

print("Out-degree  -> mean: {:.2f}, max: {}".format(out_degree.float().mean().item(), out_degree.max().item()))
print("In-degree   -> mean: {:.2f}, max: {}".format(in_degree.float().mean().item(), in_degree.max().item()))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, deg, title, color in zip(
    axes, [in_degree, out_degree], ["In-degree (times cited)", "Out-degree (papers cited)"],
    ["#DD8452", "#4C72B0"]
):
    values, counts = np.unique(deg.numpy(), return_counts=True)
    ax.loglog(values[values > 0], counts[values > 0], "o", markersize=3, color=color, alpha=0.7)
    ax.set_xlabel("Degree (log scale)")
    ax.set_ylabel("Number of papers (log scale)")
    ax.set_title(title)
    ax.grid(alpha=0.3, which="both")
plt.suptitle("Degree Distribution — OGBN-Arxiv Citation Network")
plt.tight_layout()
plt.savefig(os.path.join(ARTIFACT_DIR, "degree_distribution.png"), dpi=150)
plt.show()


### 2.6 Graph Density Analysis

**Density** measures how close the graph is to having every possible edge (density = 1 means a
complete graph). For a directed graph with `N` nodes, the maximum possible number of edges is
`N * (N-1)`. Citation networks are always extremely sparse — this number will be tiny — which is
*exactly why* GNNs (which exploit sparsity) are far more efficient here than a dense
fully-connected neural network would be.

In [ ]:
# 2.6 Graph density analysis ------------------------------------------------------------------
N = data.num_nodes
E = data.num_edges
max_possible_edges = N * (N - 1)
density = E / max_possible_edges

print(f"Nodes (N): {N:,}")
print(f"Edges (E): {E:,}")
print(f"Maximum possible directed edges (N * (N-1)): {max_possible_edges:,}")
print(f"Graph density: {density:.8f}  ({density*100:.6f}%)")
print(f"Average out-degree (E / N): {E / N:.2f}")
print("\nInterpretation: the graph is extremely sparse — a randomly chosen pair of papers has "
      f"roughly a {density*100:.6f}% chance of one citing the other. This sparsity is exactly "
      "what makes message-passing GNNs efficient: each node only aggregates from its small set "
      "of actual neighbours, never from all 169,000 other papers.")


### 2.7 Connected Component Analysis

A **connected component** is a maximal set of nodes that can all reach each other. There are two
versions of this for a *directed* graph like a citation network:

- **Weakly connected components** ignore edge direction — this is what "one giant component"
  usually refers to.
- **Strongly connected components** respect edge direction — a strong component only groups
  papers that can reach *each other* by following citations forwards. Since you can only cite
  older papers, but real citation data can contain same-year links, corrections, and metadata
  effects that create directed cycles. A strong component with more than one paper means those
  papers are mutually reachable by directed paths; it does not necessarily imply direct reciprocal citations.

We compute both with `scipy.sparse.csgraph`, which is far faster than a pure-Python NetworkX loop
at this graph's scale (1.1M edges).

In [ ]:
# 2.7 Connected component analysis (weak AND strong) --------------------------------------------
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components as scipy_connected_components

n_nodes = data.num_nodes
adj_sparse = csr_matrix(
    (np.ones(len(edge_index_np[0]), dtype=np.int8), (edge_index_np[0], edge_index_np[1])),
    shape=(n_nodes, n_nodes),
)

# Weakly connected components: ignore edge direction.
n_weak, weak_labels = scipy_connected_components(adj_sparse, directed=False)
weak_sizes_sorted = np.sort(np.bincount(weak_labels))[::-1]

# Strongly connected components: respect citation direction.
n_strong, strong_labels = scipy_connected_components(adj_sparse, directed=True, connection="strong")
strong_sizes_sorted = np.sort(np.bincount(strong_labels))[::-1]

print(f"Weakly connected components (ignoring edge direction):   {n_weak:,}")
print(f"  Largest weak component: {weak_sizes_sorted[0]:,} papers "
      f"({weak_sizes_sorted[0] / n_nodes * 100:.2f}% of the whole graph)")
print(f"\nStrongly connected components (respecting citation direction): {n_strong:,}")
print(f"  Largest strong component: {strong_sizes_sorted[0]:,} papers")
print(f"  Strong components with more than 1 paper (contain directed cycles): "
      f"{(strong_sizes_sorted > 1).sum():,}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

if n_weak == 1:
    axes[0].bar(["Entire graph"], [weak_sizes_sorted[0]], color="#55A868")
    axes[0].set_title("Weakly Connected Components\n(the whole graph is ONE component)")
    axes[0].set_ylabel("Number of papers")
else:
    top_k = min(15, len(weak_sizes_sorted))
    axes[0].bar(range(top_k), weak_sizes_sorted[:top_k], color="#55A868")
    axes[0].set_yscale("log")
    axes[0].set_xlabel("Component rank")
    axes[0].set_ylabel("Number of papers (log scale)")
    axes[0].set_title(f"{top_k} Largest Weak Components")

top_k_strong = min(20, len(strong_sizes_sorted))
axes[1].bar(range(top_k_strong), strong_sizes_sorted[:top_k_strong], color="#C44E52")
axes[1].set_yscale("log")
axes[1].set_xlabel("Component rank")
axes[1].set_ylabel("Number of papers (log scale)")
axes[1].set_title(f"{top_k_strong} Largest Strong Components\n(directed reachability structure)")

plt.tight_layout()
plt.savefig(os.path.join(ARTIFACT_DIR, "connected_components.png"), dpi=150)
plt.show()

# Keep these names for Section 8's dashboard export.
components = list(range(n_weak))
component_sizes = weak_sizes_sorted.tolist()


### 2.8 Prepare an Undirected Message-Passing Graph

Tasks 2.1-2.7 above describe the **original directed citation network**. For model training,
we preserve that directed edge list separately and convert the working `data.edge_index` to
an undirected message-passing graph. This lets each paper aggregate information from both
papers it cites and papers that cite it, matching the stronger experimental run.

The distinction is retained throughout the notebook:

- `directed_edge_index` / `directed_num_edges`: original citation relationships used in graph statistics.
- `data.edge_index` / `message_passing_num_edges`: undirected edges used by GCN, GAT, the Transformer, explainability, and live inference.

> **Rerun status:** outputs currently stored below this point belong to the earlier directed run.
> Run all cells in Colab with a GPU to replace them and regenerate consistent artifacts.

In [ ]:
# 2.8 Preserve the directed graph, then create the model's undirected graph ----------------
from torch_geometric.utils import to_undirected

directed_edge_index = data.edge_index.clone()
directed_num_edges = int(directed_edge_index.shape[1])

data.edge_index = to_undirected(directed_edge_index, num_nodes=data.num_nodes)
message_passing_num_edges = int(data.edge_index.shape[1])

print(f"Original directed citation edges: {directed_num_edges:,}")
print(f"Undirected message-passing edges: {message_passing_num_edges:,}")
assert directed_num_edges == 1_166_243
assert message_passing_num_edges == 2_315_598


---
## 3. Task 03 — Graph Data Preparation

> **Owner: Member 2** — official splits, masks and leakage-safe feature preparation.

With the raw graph loaded and understood, we now prepare it for training: load features and
labels into clean tensors, use the official OGB train/validation/test split (so results are
comparable across every group and against the OGB public leaderboard), and normalize the node
features. Every preprocessing decision is explained inline.

### 3.1 Load Node Features and Labels

`data.x` already holds the 128-D node features and `data.y` holds each paper's subject-area label
(0–39). We squeeze `y` from shape `[N, 1]` to `[N]` since PyTorch's loss functions expect flat
class-index labels, not a column vector.

### 3.2 Create Training, Validation, and Test Splits

We use the **official OGB split** rather than a random split, for two reasons: (1) it is a
realistic *temporal* split — training papers are older, test papers are the most recent, which
mirrors the real task of classifying a brand-new paper using only older citation context — and
(2) it makes our results directly comparable to every other model on the public OGB leaderboard.

### 3.3 Normalize Features

The raw 128-D features are pre-computed Word2Vec averages. We inspected their scale in Section
2.4/3.1: row norms average **2.64** with a small standard deviation (**0.24**), so they are
already reasonably well-scaled — but individual feature *columns* can still have different
scales, which can slow down or destabilize gradient descent.

We therefore **standardize each feature column** (subtract the mean, divide by the standard
deviation) rather than force-normalizing every row to unit length. Two important details:

- **Per-column, not per-row:** row-wise L2 normalization would throw away the magnitude of each
  paper's embedding, which empirically carries real predictive signal for this dataset (we
  verified this: it costs several points of test accuracy). Per-column standardization keeps that
  signal while still giving every input dimension a comparable scale for the optimizer.
- **Statistics fitted on the training set only:** we compute the mean/std from `train_idx` alone,
  never from validation or test nodes, and then apply those same statistics to the whole graph.
  This avoids leaking information from the evaluation splits into preprocessing — the correct,
  standard practice for any train/valid/test pipeline.

---
## 4. Task 04 — Graph Neural Network Development

> **Shared architecture section:** Member 2 owns GCN; Member 3 owns GAT.

We implement **two** GNN architectures as required: a **Graph Convolutional Network (GCN)** as
Model 1, and a **Graph Attention Network (GAT)** as Model 2. Both use three message-passing
layers, approximately equal parameter counts, and the same training protocol. They retain
architecture-appropriate details (GCN uses ReLU + BatchNorm; GAT uses ELU and attention
dropout), so comparisons are controlled for scale and training budget but not every operation.

### 4.1 Model 1 — Graph Convolutional Network (GCN)

**How GCN works:** each layer updates every node's representation by averaging (a symmetrically-
normalized average, to be precise) its own features with its immediate neighbours' features, then
applies a learnable linear transform and a non-linearity. Stacking `L` layers lets information
flow `L` hops across the citation graph, so a 3-layer GCN lets each paper "see" its citations'
citations' citations.

**Design choices explained:**
- **Layers:** 3 `GCNConv` layers — deep enough to reach 3-hop neighbours (common wisdom for
  citation graphs), while avoiding the well-known GNN "over-smoothing" problem that appears with
  many more layers.
- **Hidden dimension:** 256 — large enough to capture the 40-way classification signal from a
  128-D input without an excessive parameter count for a CPU/GPU laptop to train.
- **Activation function:** ReLU between layers — the standard default; it is cheap and avoids
  vanishing gradients better than sigmoid/tanh in deep stacks.
- **Regularization:** `BatchNorm1d` after each hidden layer to stabilize training, and `Dropout`
  (p=0.5) to reduce overfitting, both applied only during training.

**Note on `cached`:** `GCNConv` supports `cached=True`, which stores the normalized adjacency after the first forward pass so it does not need to be recomputed on every call — a nice speed-up when the graph never changes. We deliberately use `cached=False` here instead: Section 7.3 (Neighbourhood Influence Analysis) repeatedly feeds this same GCN a *modified* `edge_index` with individual neighbours removed, and with `cached=True` the layer would silently keep using the original, un-pruned adjacency for that computation, making the whole ablation experiment meaningless. `cached=False` costs a small amount of extra compute per forward pass but guarantees every forward pass — training or explainability — actually uses the `edge_index` it is given.


### 4.2 Model 2 — Graph Attention Network (GAT)

**How GAT works:** instead of a fixed, structurally-determined averaging weight for every
neighbour (as GCN uses), GAT *learns* an attention score for each neighbour — some cited papers
are more relevant to a paper's topic than others, and GAT lets the model discover which. Multiple
independent "attention heads" run in parallel and their outputs are concatenated (hidden layers)
or averaged (final layer), similar in spirit to multi-head attention in Transformers.

**Design choices explained:**
- **Layers:** 3 `GATConv` layers, matching GCN's depth for a fair comparison in Task 06.
- **Attention heads:** 8 heads in the hidden layers (richer representational capacity), reduced to
  1 head (averaged) on the final layer since we need exactly `num_classes` output logits, not
  `heads * num_classes`.
- **Hidden dimension:** 32 *per head* x 8 heads = 256 effective hidden width, matching GCN's 256
  for a like-for-like comparison.
- **Activation function:** ELU (the architecture's original paper's choice) between layers,
  which — unlike ReLU — has a smooth negative branch that tends to help GAT specifically converge
  more stably.
- **Regularization:** Dropout (p=0.5) applied both to node features *and* to attention
  coefficients (the latter is a GAT-specific regularization the original paper introduces).

### 4.4 Fair-Comparison Check — Parameter Counts

The GCN (hidden dimension 256) and GAT (32 dimensions x 8 heads = 256 effective width) configurations were deliberately chosen to have approximately equal trainable parameter counts. This matters for Task 06: if one model simply had far more parameters than the other, a difference in accuracy could be explained by raw capacity rather than by the architectural idea (fixed neighbourhood averaging vs. learned attention) being compared.


### 4.3 Move the Graph to the Training Device

Full-batch training is used (the whole graph and both models fit comfortably in memory for
169k nodes), so we move the graph's tensors to `DEVICE` once, here, rather than inside the
training loop.

---
## 5. Task 05 — Model Training and Optimization

> **Owner: Member 3** — optimisation, early stopping, sensitivity study and training logs.

Both models are trained with the same protocol so their eventual scores in Task 06 are comparable:
same loss function, same optimizer family, same number of epochs, and the same early-stopping rule
(keep the checkpoint with the best validation accuracy seen so far).

### 5.1 Loss Function and Optimizer

- **Loss function: Cross-Entropy Loss.** This is the standard choice for multi-class
  classification with mutually-exclusive classes (each paper has exactly one subject area) — it
  directly penalizes low predicted probability on the true class.
- **Optimizer: Adam.** Adam adapts the learning rate per-parameter using running estimates of the
  gradient's mean and variance, which converges faster and more reliably than plain SGD on GNNs,
  where gradient magnitudes vary a lot between low-degree and high-degree nodes.
- **Learning rate: `0.01`.** This matches the setting used in the original GCN paper and in most
  published OGBN-Arxiv baselines. *(An earlier draft of this notebook used `0.005` combined with
  a low epoch budget, which under-fit and plateaued around 57-60% validation accuracy — well
  below what these architectures are capable of. Raising the learning rate and, more importantly,
  giving training more epochs to actually converge — see below — closes most of that gap.)*
- **Weight decay (L2 regularization):** `5e-4`, added directly inside the optimizer, to further
  discourage over-fitting on top of Dropout.
- **Epoch budget:** up to `500` epochs with early stopping (patience `40` epochs with no
  validation-accuracy improvement) — enough headroom for full-batch training on this graph size to
  actually reach its plateau, rather than stopping while still climbing.

### 5.2 Hyperparameters Considered

Rather than only listing candidate values (which nobody reading the notebook can verify), the cell below actually **runs each candidate** as a short, throwaway training run (60 epochs, no early stopping — long enough to compare configurations, short enough to keep this cell fast) and records its best **validation** accuracy. Hyperparameters are selected on validation accuracy only; the test set is never touched here and is evaluated exactly once, later, in Task 06.

The search varies learning rate, number of layers, hidden dimensions and dropout — the four hyperparameters the brief explicitly names for Task 05. Weight decay (`5e-4`) and the Adam optimizer are kept fixed throughout the search (both are well-established defaults for GNNs) so that the search isolates the effect of the other four.


### 5.2b Interpreting the Short Search

The 60-epoch search is a controlled **sensitivity check**, not a substitute for the final training protocol. Its best row can change after converting the message-passing graph to undirected form. The final GCN and GAT therefore retain the coursework's parameter-matched three-layer designs and are trained with early stopping for a fair model comparison. The test split remains untouched during selection.

### 5.3 Training Loop Function

A single reusable `train_model` function is defined so both GCN and GAT are trained through
identical code — this removes any risk of accidentally giving one model an unfair advantage. Every
epoch we: run a forward pass on the *full graph*, compute loss only on `train_mask` nodes, back-
propagate, then evaluate accuracy on `train_mask` and `valid_mask` (without dropout, using
`model.eval()`). The best-validation-accuracy checkpoint is kept, and training stops early if
validation accuracy has not improved for `PATIENCE` epochs.

### 5.4 Train the GCN Model

### 5.5 Train the GAT Model

### 5.6 Monitor Training Performance

We plot loss and accuracy curves for both models side by side. A healthy run shows training loss
falling steadily and validation accuracy flattening out (rather than dropping, which would signal
over-fitting) near the early-stopping point.

### 5.7 Save Trained Model Weights

We persist both models' trained weights to disk — this is one of the required submission items
("Trained Model Files") and also what the Streamlit dashboard (Task 08) loads to serve live
predictions without needing to retrain.

---
## 6. Task 06 — Model Evaluation

> **Owner: Member 4** — common evaluation metrics and controlled model comparison.

We now evaluate both trained models on the held-out **validation** and **test** sets using four
standard classification metrics: Accuracy, Precision, Recall, and F1 Score. Since this is a
40-class problem with an imbalanced class distribution (Section 3.1), we report **macro-averaged**
Precision/Recall/F1 (every class weighted equally) alongside plain Accuracy, so that performance on
small/rare subject areas is visible and not hidden by the large classes.

### 6.1 Evaluation Function

One shared function computes all four metrics for a given model + mask, so GCN and GAT are scored
identically.

### 6.2 Evaluate Both Models on Validation and Test Sets

### 6.2b Official OGB Benchmark Accuracy

`accuracy_score` from scikit-learn (used above) computes plain top-1 accuracy, which for `ogbn-arxiv` is numerically identical to the official OGB evaluator's accuracy metric. We additionally report it through OGB's own `Evaluator` here so the results are presented in the benchmark-standard way, directly comparable to the public OGBN-Arxiv leaderboard.


### 6.3 Visual Comparison of Both Models

---
## 7. Task 07 — Graph Explainability and Embedding Analysis

> **Owner: Member 4** — embeddings, attention interpretation and neighbourhood ablation.

The brief asks for at least **two** of: Feature Importance, Node Embedding Visualization,
Attention Weight Analysis, or Neighbourhood Influence Analysis. We implement **three** for a more
complete picture of *why* the models predict what they predict: (A) embedding visualization with
t-SNE, (C) GAT attention-weight analysis, and (D) neighbourhood influence analysis.

### 7.1 Option B — Node Embedding Visualization (t-SNE)

We extract the 256-dimensional representation each model computes just *before* its final
classification layer (the learned "embedding" of each paper) and project it down to 2 dimensions
with t-SNE. If the model has learned something meaningful, papers from the same subject area
should visually cluster together — this is one of the most intuitive ways to *see* what a GNN has
learned.

**Interpreting the t-SNE / PCA projections:** the t-SNE projection above should show several locally concentrated clusters that broadly correspond to subject-area classes, which is evidence that the GCN's learned embeddings carry class-discriminative information rather than being unstructured noise. Expect some visible overlap between neighbouring clusters too — this is consistent with the model's sub-100% accuracy, and likely corresponds to computer-science subject areas that are genuinely related (sharing both textual features and citation patterns, e.g. `cs.LG` and `cs.AI`), so some confusion between them is reasonable rather than a modelling failure. The PCA projection is included for comparison (Option B explicitly allows PCA *or* t-SNE): PCA typically shows the same broad class groupings with less fine-grained cluster separation, since it is restricted to a single global linear projection rather than t-SNE's local, non-linear one.


### 7.2 Option C — Attention Weight Analysis (GAT)

GAT computes an explicit attention score for every edge at every layer, telling us exactly how much weight the model places on each cited paper when forming a node's new representation.

**Choosing a good example paper matters here.** If we picked a paper with hundreds of citers, softmax-normalized attention naturally flattens out to roughly `1 / degree` for every neighbour — technically correct, but a visually uninformative "every bar is the same height" chart. We select **test-set** papers (nodes the model never trained on) with a **moderate** number of citers (5–25), where individual neighbours can meaningfully stand out from each other: one the model classifies **correctly**, reused in Section 7.3, and — where one exists among the moderate-degree test candidates — one it classifies **incorrectly**, for contrast. Explaining a test prediction (rather than a training node) demonstrates how the trained model actually behaves on unseen labels.

**Self-loops:** PyG's `GATConv` adds a self-loop to every node by default, so a node's own ID will appear in its own incoming-edge list. That self-loop is the model attending to the paper's own features, not a citation relationship, so it is excluded from the "top attended neighbours" table below and reported separately instead.


### 7.2b [Extended] Does High Attention Correspond to Same-Subject Citations?

Looking at one node's top attended neighbours is illustrative but anecdotal. To answer the
underlying research question properly, we quantify it across **all test-set edges**: for the
top-K highest-attention citation edges (globally, not just for one paper), what fraction connect
two papers of the *same* subject class - compared with the same statistic computed over a random
sample of citation edges? If attention is doing something meaningful, the high-attention group
should have a noticeably higher same-class rate than the random baseline.


### 7.3 Option D — Neighbourhood Influence Analysis

A complementary, model-agnostic way to explain a prediction: **remove each 1-hop neighbour one at
a time and re-run the model**, measuring how much the target node's predicted class probability
shifts. Neighbours whose removal causes the biggest probability drop are, empirically, the most
influential for that specific prediction — this does not depend on GAT's attention mechanism, so
it can sanity-check the attention-based explanation above and can be applied to GCN too.

### 7.4 Cross-Check — Does GAT Attention Agree With Ablation Influence?

Sections 7.2 and 7.3 explain the *same* target paper using two independent, model-agnostic methods: GAT's learned attention weights, and GCN's leave-one-neighbour-out probability drop. They come from two separately-trained models, so exact agreement is not expected — but if the two top-10 neighbour lists below overlap more than chance would predict, that is evidence both methods are picking out genuinely important citations rather than noise.


---
## 8. Task 08 — Export Artifacts for the Graph Intelligence Dashboard

> **Owner: Member 5 / Project Owner (you)** — dashboard prototype and artifact export.

The Streamlit dashboard (`app.py`, submitted alongside this notebook — see the accompanying
`README.md` for how to launch it) is a **separate application** so that it starts instantly and
never needs to re-run training. This final section exports everything the dashboard needs as small
JSON/CSV/NumPy files into `dashboard/artifacts/`: graph statistics, both models' metrics, sample
predictions, and 2-D embedding coordinates for the interactive scatter plot.

### 8.1b Export the Real Subject-Area Class Names

Every class ID we've printed so far (0-39) is just an internal index — it doesn't say *which*
arXiv CS subject area it actually is. OGB ships the true mapping alongside the dataset itself
(`data/ogbn_arxiv/mapping/labelidx2arxivcategeory.csv.gz`), so we load it from disk here and
export it for the dashboard — this is the **only correct source** for these names (guessing or
hard-coding a plausible-looking list risks silently mislabelling every prediction in the UI).

### 8.2 Export Model Performance Metrics

Both models' validation/test metrics from Section 6, plus parameter counts and training time from
Section 5, are written to a single JSON file — this is what the dashboard's "Model Performance"
tab reads to draw the GCN-vs-GAT comparison charts without needing PyTorch or the trained model
files loaded at all.

### 8.3 Export Node Classification Results

We export a sample of individual predictions — each paper's true label, both models' predicted
label, their confidence, and whether they were correct — as a CSV. This is the "results" table the
brief's dashboard requirement asks for, letting a reader browse individual predictions rather than
only seeing aggregate metrics.

### 8.4 Export 2-D Embedding Coordinates

Finally, we export the t-SNE and PCA coordinates computed back in Section 7.1 (plus each sampled
paper's true class) as a small CSV. This is what powers the dashboard's *interactive* embedding
scatter plot — letting the reader zoom, pan, and hover over individual papers — instead of the
notebook's static, non-interactive image.

---
## Extended — Graph Transformer

> **Owner: Member 5 / Project Owner (you)** — optional Transformer/SSL work.

This section is **not required** for the core pipeline above (Tasks 01–09 are fully self-contained
without it). It explores "Graph Transformers" and "Advanced GNN Architectures" as an optional
extension. It is kept in its own clearly separated section, after every core task, so the main
submission can be assessed on its own and everything below treated as extra.

**Model:** a 3-layer Graph Transformer built from PyTorch Geometric's `TransformerConv`, which generalises GAT's attention to a full Transformer-style multi-head query/key/value attention **and** an optional learned edge-feature term, over the same citation graph. Depth (3 layers), hidden width (32 x 8 heads = 256), and heads (8) are kept identical to the GAT baseline in Section 4.2 so the three-way comparison below isolates the effect of the attention mechanism itself rather than differences in model capacity.



### Extended.2 Train the Graph Transformer

Trained with the exact same protocol as GCN/GAT (`train_model` from Section 5.3: Adam, the same learning rate/weight decay, the same max-epoch budget and early-stopping patience) so its results in the comparison below are directly comparable, not an apples-to-oranges training setup.


### Extended.3 Evaluate and Compare All Three Models


---
## Extended — Self-Supervised Pre-training

> **Owner: Member 5 / Project Owner (you)** — optional Transformer/SSL work.

Also **not required** for the core pipeline (Tasks 01-09 are fully self-contained without
it). It explores "self-supervised graph pre-training" as an optional extension.

**Idea:** pre-train a GCN encoder to reconstruct **masked node features** using only the graph
structure and features - no class labels at all. The encoder is then reused as the initialisation
for a supervised classifier, which is fine-tuned on the official labelled training split. This
tests whether structure-aware pre-training gives the model a better starting point than training
from a random initialisation.

**Avoiding label leakage / split contamination:** the reconstruction (pretext) task masks and
reconstructs features for a **random sample of ALL nodes in the graph** (train, validation, and
test alike), not just the training split - no class labels are used anywhere in this stage, so
which nodes get masked has no bearing on the official split. The official train/validation/test
masks are touched for the very first time only in the fine-tuning stage below, exactly as with the
supervised GCN/GAT/Transformer models above.


### Extended-SSL.2 Pretext Task: Masked Feature Reconstruction

Each epoch we randomly mask **15%** of node feature vectors (replacing them with zeros, sampled
from *all* nodes - train, validation, and test alike) and train the encoder + a reconstruction
head to predict the original feature vector from graph context alone, minimising mean-squared
error only on the masked entries. No labels are used at any point in this stage.


### Extended-SSL.3 Fine-tune on the Official Labelled Split, and Compare Against a From-Scratch Baseline

This is the first point in the SSL pipeline where labels or the official train/validation/test
masks are used. We fine-tune **two** classifiers with the exact same architecture and training
protocol (`train_model` from Section 5.3):

1. **SSL-pretrained** - initialised from the masked-feature-reconstruction encoder above.
2. **From-scratch baseline** - identical architecture, random initialisation, no pretraining.

Comparing these two isolates the effect of the self-supervised pretext task from the effect of
the architecture itself.


---
## 9. Task 09 — Five-Member Contribution and Viva Map

| Member | Assigned notebook work | Git branch |
|---|---|---|
| Member 1 | Tasks 01–02: tensors, dataset loading, graph representation and analysis | `feature/member-1-tensors-graph` |
| Member 2 | Task 03 and GCN: preparation, masks, normalisation and GCN architecture | `feature/member-2-data-gcn` |
| Member 3 | GAT and Task 05: attention architecture, optimisation, training and checkpoints | `feature/member-3-gat-training` |
| Member 4 | Tasks 06–07: evaluation, comparison, embeddings and explainability | `feature/member-4-evaluation` |
| Member 5 — Project Owner | Task 08, prototype, extended work and final integration | `feature/owner-prototype-integration` |

Each member adds code cells only inside their assigned section. The project owner reviews and
merges branches, resolves notebook conflicts, runs the final notebook in Colab, and regenerates
the dashboard artifacts. Every member must understand the shared pipeline for the viva.


---
## Final Colab Artifact Download

After all core and extended experiments have finished, this creates one ZIP containing the complete regenerated dashboard artifact directory. In Google Colab the download starts automatically; locally, the absolute ZIP path is printed. Replace the project's existing `dashboard/artifacts/` only after this run completes successfully.

---
## Notebook Complete

This notebook contains the computational implementation for Tasks 1–7, plus clearly separated extended experiments (Graph Transformer and self-supervised pre-training), and exports the artifacts the Streamlit dashboard (Task 08) reads (`dashboard/artifacts/`). The dashboard source is kept separately in `dashboard/app.py`. The Technical Report (PDF), presentation slides, GitHub repository, and 5–10 minute video demonstration are also separate required submission components, listed in the brief's "Submission Requirements" section.


---
## 10. Download Everything as a ZIP

> **Owner: Member 5 / Project Owner (you)** — final integration and submission packaging.

> **Run this section locally from the organised project root, not in Colab.** Colab does not contain the saved canonical notebook, dashboard application, report, or presentation files required for the final submission package.

This final section packages every deliverable this notebook produced — the trained model files,
all exported charts/images, the metrics/predictions/embeddings CSVs and JSONs, and (if they sit
the organised `notebook/` and `dashboard/` folders, root documentation, and (when available)
`Technical_Report.pdf` — into a single ZIP file ready to submit or upload.